<a href="https://colab.research.google.com/github/mdkamrulhasan/machine_learning_concepts/blob/master/notebooks/supervised/Regression_Insurance_HpOpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyper-parameter Optimization: Decision Tree vs. Random Forest

Predict medical insurance `charges` with two tree models, tune their hyper-parameters with grid search, and compare them **before and after** tuning.

* **Decision Tree:** tune `max_depth`
* **Random Forest:** tune `max_depth` and `n_estimators`

Error metric: Mean Absolute Error (MAE, in dollars; lower is better).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error

## 1. Data
[Medical insurance cost dataset (src)](https://www.kaggle.com/datasets/mirichoi0218/insurance/)

In [ ]:
data_url = 'https://raw.githubusercontent.com/mdkamrulhasan/data_mining_kdd/main/data/medical-cost/insurance.csv'
df = pd.read_csv(data_url)
df.head()

One-hot encode the categorical features (`sex`, `smoker`, `region`). Trees split on one feature at a time, so **no scaling is needed**.

In [ ]:
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True, dtype=int)
y = df['charges']

# Hold out a test set; all tuning uses cross-validation on the training set only
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=0)
print(X.columns.tolist())
print('train:', X_train.shape, ' test:', X_test.shape)

## 2. Before tuning: default hyper-parameters

In [ ]:
def cv_mae(model):
    # 5-fold cross-validated MAE on the training set
    return -cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')

dt_default = DecisionTreeRegressor(random_state=0)
rf_default = RandomForestRegressor(random_state=0)   # n_estimators=100, max_depth=None

results = {}
for name, model in [('Decision Tree', dt_default), ('Random Forest', rf_default)]:
    scores = cv_mae(model)
    model.fit(X_train, y_train)
    results[(name, 'Default')] = dict(cv=scores, test=mean_absolute_error(y_test, model.predict(X_test)))
    print(f'{name:14s} default: CV MAE = {scores.mean():,.0f} ± {scores.std():,.0f}')

print('Default tree depth:', dt_default.get_depth())   # grows until leaves are pure -> overfits

## 3. Tune the Decision Tree: `max_depth`

In [ ]:
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=0),
                       param_grid={'max_depth': list(range(1, 16))},
                       cv=cv, scoring='neg_mean_absolute_error', return_train_score=True)
dt_grid.fit(X_train, y_train)
print('Best:', dt_grid.best_params_, f' CV MAE = {-dt_grid.best_score_:,.0f}')

## 4. Tune the Random Forest: `max_depth` × `n_estimators`

In [ ]:
rf_grid = GridSearchCV(RandomForestRegressor(random_state=0, n_jobs=-1),
                       param_grid={'max_depth': [2, 3, 4, 5, 6, 8, 10, 15],
                                   'n_estimators': [10, 25, 50, 100, 200]},
                       cv=cv, scoring='neg_mean_absolute_error')
rf_grid.fit(X_train, y_train)
print('Best:', rf_grid.best_params_, f' CV MAE = {-rf_grid.best_score_:,.0f}')

In [ ]:
# Record the tuned models (best_estimator_ is refit on the full training set)
for name, grid in [('Decision Tree', dt_grid), ('Random Forest', rf_grid)]:
    best = grid.best_estimator_
    results[(name, 'Tuned')] = dict(cv=cv_mae(best),
                                    test=mean_absolute_error(y_test, best.predict(X_test)))

## 5. Compare the models

### 5a. Effect of tree depth
Cross-validated MAE as `max_depth` grows. For the Random Forest, one line per `n_estimators`.

In [ ]:
dt_cv = pd.DataFrame(dt_grid.cv_results_)
rf_cv = pd.DataFrame(rf_grid.cv_results_)

fig = go.Figure()
fig.add_scatter(x=dt_cv.param_max_depth, y=-dt_cv.mean_test_score, mode='lines+markers',
                name='Decision Tree', line=dict(width=4, color='black'))
for n, g in rf_cv.groupby('param_n_estimators'):
    fig.add_scatter(x=g.param_max_depth, y=-g.mean_test_score, mode='lines+markers',
                    name=f'RF, {n} trees')
fig.update_layout(title='CV MAE vs. tree depth', xaxis_title='max_depth',
                  yaxis_title='CV MAE ($)', template='plotly_white')
fig.show()

The Decision Tree's error falls and then **rises again** as depth increases (overfitting). Averaging many trees makes the Random Forest much less sensitive to depth.

### 5b. Random Forest search space

In [ ]:
heat = rf_cv.pivot(index='param_n_estimators', columns='param_max_depth', values='mean_test_score')
fig = px.imshow(-heat, text_auto=',.0f', color_continuous_scale='Viridis_r', aspect='auto',
                labels=dict(x='max_depth', y='n_estimators', color='CV MAE ($)'),
                title='Random Forest: CV MAE for each (max_depth, n_estimators)')
fig.update_yaxes(type='category'); fig.update_xaxes(type='category')
fig.show()

Depth matters far more than the number of trees: beyond roughly 50 trees, adding more barely changes the error.

### 5c. Before vs. after tuning

In [ ]:
summary = pd.DataFrame([
    dict(Model=m, Setting=s, CV_MAE=r['cv'].mean(), CV_SD=r['cv'].std(), Test_MAE=r['test'])
    for (m, s), r in results.items()])
display(summary.round(0))

fig = px.bar(summary, x='Model', y='CV_MAE', color='Setting', barmode='group', error_y='CV_SD',
             text_auto=',.0f', title='CV MAE (training set) before vs. after tuning',
             labels={'CV_MAE': 'MAE ($)'}, template='plotly_white')
fig.show()

fig = px.bar(summary, x='Model', y='Test_MAE', color='Setting', barmode='group',
             text_auto=',.0f', title='Held-out test MAE before vs. after tuning',
             labels={'Test_MAE': 'MAE ($)'}, template='plotly_white')
fig.show()

In [ ]:
for m in ['Decision Tree', 'Random Forest']:
    before, after = results[(m, 'Default')]['test'], results[(m, 'Tuned')]['test']
    print(f'{m:14s} test MAE: {before:,.0f} -> {after:,.0f}  (change {100 * (after / before - 1):+.1f}%)')
print('Best Decision Tree:', dt_grid.best_params_)
print('Best Random Forest:', rf_grid.best_params_)

### Takeaways
* **Tuning matters most for the Decision Tree.** The default tree grows until every leaf is pure (depth ≈ 20) and overfits. Limiting `max_depth` gives a clear improvement.
* **The Random Forest is already robust with defaults.** Averaging many trees controls overfitting, so tuning gives a small CV gain and little to no gain on the test set.
* **The Random Forest beats the Decision Tree** both before and after tuning.
* A CV score of the *selected* setting is slightly optimistic (the same folds chose it); the held-out test MAE is the honest estimate.